# 02 - Embeddings Profundos com CelebA Align e StyleGAN2

Neste notebook vamos gerar embeddings com `ResNet50`, visualizar a distribuicao em 2D e salvar o modelo padrao considerando **as imagens originais e as imagens sinteticas**.


In [1]:
import pickle
from math import ceil
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50

try:
    import umap
except ImportError:
    umap = None

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATASET_ROOT = Path('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba')
SYNTHETIC_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'stylegan2_synthetic_256'
OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'embeddings' / 'celeba_align_plus_synthetic_deep_resnet50.parquet'
MODEL_OUTPUT_PATH = PROJECT_ROOT / 'artifacts' / 'models' / 'deep_resnet50_model.pkl'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
SAMPLE_FRACTION = 0.15
SAMPLE_RANDOM_SEED = 42
MAX_SAMPLES_PER_SOURCE = None
IMAGE_SIZE = 256

for folder in [OUTPUT_PATH.parent, MODEL_OUTPUT_PATH.parent, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Originais em:', DATASET_ROOT)
print('Sinteticas em:', SYNTHETIC_ROOT)
print('Fracao da amostra:', SAMPLE_FRACTION)
print('Sample random seed:', SAMPLE_RANDOM_SEED)


Originais em: /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/img_align_celeba
Sinteticas em: /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/processed/stylegan2_synthetic_256
Fracao da amostra: 0.15
Sample random seed: 42


In [3]:
def build_combined_manifest(original_root, synthetic_root=None, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None):
    sources = [
        ('original', Path(original_root), '*.jpg'),
        ('synthetic', Path(synthetic_root), '*.png'),
    ]

    frames = []
    for source_name, root, pattern in sources:
        if root is None or not root.exists():
            continue
        images = sorted(root.glob(pattern))
        if images:
            sample_size = min(len(images), ceil(len(images) * sample_fraction))
            rng = random.Random(sample_random_seed)
            images = sorted(rng.sample(images, sample_size))
        if max_samples_per_source:
            images = images[:max_samples_per_source]
        if images:
            frames.append(pd.DataFrame({
                'image_name': [path.name for path in images],
                'image_path': [str(path) for path in images],
                'source': source_name,
            }))

    if not frames:
        raise FileNotFoundError('Nenhuma imagem original ou sintetica foi encontrada.')

    return pd.concat(frames, ignore_index=True)

class AlignCelebADataset(Dataset):
    def __init__(self, original_root, synthetic_root=None, image_size=256, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None):
        self.samples = build_combined_manifest(
            original_root=original_root,
            synthetic_root=synthetic_root,
            sample_fraction=sample_fraction,
            sample_random_seed=sample_random_seed,
            max_samples_per_source=max_samples_per_source,
        )
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        row = self.samples.iloc[index]
        image = Image.open(row['image_path']).convert('RGB')
        return {
            'image': self.transform(image),
            'image_name': row['image_name'],
            'image_path': row['image_path'],
            'source': row['source'],
        }

def build_resnet50_embedding_model():
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Identity()
    model.eval()
    return model

def extract_deep_embeddings(original_root, synthetic_root, output_path, batch_size=32, num_workers=0, sample_fraction=0.20, sample_random_seed=42, max_samples_per_source=None, image_size=256, device=None):
    dataset = AlignCelebADataset(
        original_root=original_root,
        synthetic_root=synthetic_root,
        image_size=image_size,
        sample_fraction=sample_fraction,
        sample_random_seed=sample_random_seed,
        max_samples_per_source=max_samples_per_source,
    )
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    resolved_device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_resnet50_embedding_model().to(resolved_device)

    rows = []
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(resolved_device)
            embeddings = model(images).cpu().numpy()
            image_names = batch['image_name']
            image_paths = batch['image_path']
            sources = batch['source']
            for embedding, image_name, image_path, source in zip(embeddings, image_names, image_paths, sources):
                row = {'image_name': image_name, 'image_path': image_path, 'source': source}
                for index, value in enumerate(embedding):
                    row[f'f_{index:04d}'] = float(value)
                rows.append(row)

    frame = pd.DataFrame(rows)
    frame.to_parquet(output_path, index=False)
    return frame

def feature_matrix(frame):
    return frame.drop(columns=['image_name', 'image_path', 'source'])

def evaluate_source_classification(frame, test_size=0.20, random_state=42):
    features = feature_matrix(frame)
    labels = (frame['source'] == 'synthetic').astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        features,
        labels,
        test_size=test_size,
        random_state=random_state,
        stratify=labels,
    )
    estimator = Pipeline([
        ('scaler', StandardScaler(with_mean=False)),
        ('classifier', LogisticRegression(max_iter=2000, random_state=random_state)),
    ])
    estimator.fit(X_train, y_train)
    predictions = estimator.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    return {
        'metric': 'source_classification',
        'accuracy': float(accuracy),
        'error_rate': float(1.0 - accuracy),
        'train_samples': int(len(X_train)),
        'test_samples': int(len(X_test)),
    }

def reduce_embeddings(frame, method='pca', random_state=42):
    features = feature_matrix(frame)
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=random_state)
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=random_state, init='pca')
    elif method == 'umap':
        if umap is None:
            raise ImportError("Instale 'umap-learn' para usar UMAP.")
        reducer = umap.UMAP(n_components=2, random_state=random_state)
    else:
        raise ValueError(f'Metodo desconhecido: {method}')
    reduced = reducer.fit_transform(features)
    result = frame[['image_name', 'image_path', 'source']].copy()
    result['x'] = reduced[:, 0]
    result['y'] = reduced[:, 1]
    result['method'] = method
    return result

def save_scatter_plot(frame, output_path, title, hue='source'):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(10, 8))
    sns.scatterplot(data=frame, x='x', y='y', hue=hue, alpha=0.75, s=40)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

def save_pickle_model(model, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'wb') as file:
        pickle.dump(model, file)
    return output_path

def save_default_resnet50_model(output_path):
    model = build_resnet50_embedding_model().cpu()
    return save_pickle_model(model, output_path)


## Extracao dos embeddings profundos

In [4]:
deep_df = extract_deep_embeddings(
    original_root=DATASET_ROOT,
    synthetic_root=SYNTHETIC_ROOT,
    output_path=OUTPUT_PATH,
    sample_fraction=SAMPLE_FRACTION,
    sample_random_seed=SAMPLE_RANDOM_SEED,
    max_samples_per_source=MAX_SAMPLES_PER_SOURCE,
    image_size=IMAGE_SIZE,
)
deep_df.head()


,image_name,image_path,source,f_0000,f_0001,f_0002,f_0003,f_0004,f_0005,f_0006,...,f_2038,f_2039,f_2040,f_2041,f_2042,f_2043,f_2044,f_2045,f_2046,f_2047
0,000003.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.000000,0.000000,0.000000,0.000424,0.015804,0.000000,0.058349,...,0.000000,0.000000,0.210535,0.265472,0.180451,0.006644,0.000060,0.001396,0.000000,0.143836
1,000010.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.053783,0.000000,0.000000,0.330517,0.000000,0.058748,0.000000,...,0.000000,0.000000,0.029333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,000016.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000989,0.088447,0.000000,0.000000,0.000000,0.008812,0.000000,0.000000,0.000000
3,000023.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.000000,0.064155,0.052262,0.498458,0.000000,0.000000,0.000000,...,0.000000,0.036913,0.023445,0.007073,0.000000,0.000000,0.410091,0.000000,0.469659,0.435902
4,000025.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.000000,0.012907,0.084922,0.002388,0.000000,0.009932,0.000000,...,0.037689,0.048192,0.072056,0.203729,0.000000,0.009797,0.020328,0.000000,0.000000,0.345933


## Reducao para 2D

In [5]:
deep_2d = reduce_embeddings(deep_df, method='pca')
deep_2d.head()


,image_name,image_path,source,x,y,method
0,000003.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.339711,0.472568,pca
1,000010.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,-4.456803,-1.377590,pca
2,000016.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,4.068021,-0.957959,pca
3,000023.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,5.199580,0.073334,pca
4,000025.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,5.595045,0.924791,pca


In [6]:
figure_path = FIGURES_DIR / 'celeba_align_deep_pca.png'
save_scatter_plot(deep_2d, figure_path, 'CelebA Align - Deep Embeddings (PCA)')
figure_path

PosixPath('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/reports/figures/celeba_align_deep_pca.png')

In [7]:
model_path = save_default_resnet50_model(MODEL_OUTPUT_PATH)
deep_metrics = evaluate_source_classification(deep_df, random_state=SAMPLE_RANDOM_SEED)
print('Modelo ResNet50 padrao salvo em:', model_path)
print('Accuracy (original vs synthetic):', round(deep_metrics['accuracy'], 4))
print('Error rate:', round(deep_metrics['error_rate'], 4))
pd.DataFrame([deep_metrics])

Modelo ResNet50 padrao salvo em: /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/artifacts/models/deep_resnet50_model.pkl
Accuracy (original vs synthetic): 1.0
Error rate: 0.0


,metric,accuracy,error_rate,train_samples,test_samples
0,source_classification,1.0,0.0,27229,6808


In [8]:
deep_df

,image_name,image_path,source,f_0000,f_0001,f_0002,f_0003,f_0004,f_0005,f_0006,...,f_2038,f_2039,f_2040,f_2041,f_2042,f_2043,f_2044,f_2045,f_2046,f_2047
0,000003.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.000000,0.000000,0.000000,0.000424,0.015804,0.000000,0.058349,...,0.000000,0.000000,0.210535,0.265472,0.180451,0.006644,0.000060,0.001396,0.000000,0.143836
1,000010.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.053783,0.000000,0.000000,0.330517,0.000000,0.058748,0.000000,...,0.000000,0.000000,0.029333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,000016.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000989,0.088447,0.000000,0.000000,0.000000,0.008812,0.000000,0.000000,0.000000
3,000023.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.000000,0.064155,0.052262,0.498458,0.000000,0.000000,0.000000,...,0.000000,0.036913,0.023445,0.007073,0.000000,0.000000,0.410091,0.000000,0.469659,0.435902
4,000025.jpg,/home/arthur/Documentos/Github/Projeto-5-Redes...,original,0.000000,0.012907,0.084922,0.002388,0.000000,0.009932,0.000000,...,0.037689,0.048192,0.072056,0.203729,0.000000,0.009797,0.020328,0.000000,0.000000,0.345933
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34032,seed9976.png,/home/arthur/Documentos/Github/Projeto-5-Redes...,synthetic,0.015189,0.002730,0.000000,0.002294,0.000000,0.008415,0.000000,...,0.000000,0.110642,0.033719,0.000000,0.000000,0.000000,0.017755,0.113719,0.010817,0.000000
34033,seed9977.png,/home/arthur/Documentos/Github/Projeto-5-Redes...,synthetic,0.149666,0.000000,0.000000,0.083898,0.000000,0.074373,0.000000,...,0.000000,0.065156,0.201935,0.025527,0.000000,0.000000,0.027647,0.031942,0.000000,0.070757
34034,seed9984.png,/home/arthur/Documentos/Github/Projeto-5-Redes...,synthetic,0.133632,0.000000,0.000000,0.000000,0.000000,0.000000,0.015041,...,0.000000,0.146821,0.260202,0.010436,0.000000,0.000000,0.004990,0.044806,0.000000,0.000000
34035,seed9987.png,/home/arthur/Documentos/Github/Projeto-5-Redes...,synthetic,0.258091,0.000000,0.019430,0.660581,0.000000,0.052815,0.008386,...,0.000000,0.341813,0.080902,0.030685,0.000000,0.049662,0.620436,0.000000,0.000000,0.027549
